# A2 — Point-in-Time Exact Retrieval Evaluation

This notebook evaluates the frozen V4 checkpoint without retraining. Every variant uses the same fixed protocol-v2 queries, time-eligible item catalog, strict-prior seen-item filter, and exact target at query time.

Compared variants:

- train-cutoff Popularity;
- V4 static ablation (same model with recent sequence removed);
- V4 adaptive;
- V4 adaptive plus the existing popularity blend.

Use an L4 or A100 GPU. No model is trained and only lightweight reports are persisted.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
PROTOCOL_DIR = ADAPTIVE_ROOT / 'evaluation_protocol/v2'
WORK_ROOT = Path('/content/recsys_protocol_v2_a2')
SOURCE_ROOT = WORK_ROOT / 'runtime'
V4_DIR = WORK_ROOT / 'artifacts/v4_sequence_retrieval'
OUTPUT_DIR = WORK_ROOT / 'reports/protocol_v2_a2'
for path in [SOURCE_ROOT, V4_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('protocol:', PROTOCOL_DIR)
print('work root:', WORK_ROOT)

In [ ]:
required = {'pyarrow': 'pyarrow', 'pandas': 'pandas', 'torch': 'torch', 'huggingface_hub': 'huggingface_hub>=0.32'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import pandas as pd
import torch
assert torch.cuda.is_available(), 'Select an L4 or A100 GPU runtime.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    PROTOCOL_DIR / 'queries.parquet',
    PROTOCOL_DIR / 'manifest.json',
    BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
    BASE_GOLD_ROOT / 'vocabularies/user_active_degree',
    BASE_GOLD_ROOT / 'vocabularies/video_type',
    BASE_GOLD_ROOT / 'vocabularies/upload_type',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing A2 inputs:\n' + '\n'.join(missing_paths)
protocol_manifest = json.loads((PROTOCOL_DIR / 'manifest.json').read_text())
assert protocol_manifest['builder_revision'] == '2.1-session-history'
print('Drive preflight: PASS')
print('queries/users:', protocol_manifest['queries'], protocol_manifest['users'])

## Download the tested runtime and frozen V4

The token is read from Colab Secrets when available. If the Secrets API times out, paste a Hugging Face read token into the hidden prompt.

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
V4_EXPECTED_SHA256 = '4ad49b3e6f9f8b5d45a6c92f31d8d9b27f558e1746613efb44a23db3cd456a26'
RUNTIME_EXPECTED_SHA256 = 'f812fd2ad652410d793cfb96c1c57c37fc9db7c6f00d76b8cf405ba576a7d337'
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception as exc:
    print('Colab Secrets unavailable:', type(exc).__name__)
    HF_TOKEN = getpass('Paste Hugging Face READ token: ')
assert HF_TOKEN and HF_TOKEN.startswith('hf_'), 'A valid Hugging Face token must start with hf_.'
runtime_zip = Path(hf_hub_download(HF_REPO_ID, 'runtime/adaptive_eval_runtime.zip', repo_type='model', token=HF_TOKEN))
v4_cached = Path(hf_hub_download(HF_REPO_ID, 'v4/v4_inference_fp16.pt', repo_type='model', token=HF_TOKEN))

def sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

assert sha256(runtime_zip) == RUNTIME_EXPECTED_SHA256, 'Runtime ZIP hash mismatch'
assert sha256(v4_cached) == V4_EXPECTED_SHA256, 'V4 checkpoint hash mismatch'
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
evaluation_script = SOURCE_ROOT / 'scripts/evaluate_adaptive_protocol_v2.py'
assert evaluation_script.exists(), f'Missing evaluator: {evaluation_script}'
v4_target = V4_DIR / 'best_model.pt'
if v4_target.exists() or v4_target.is_symlink():
    v4_target.unlink()
v4_target.symlink_to(v4_cached.resolve())
print('artifact integrity: PASS')
print('V4:', f'{v4_cached.stat().st_size / 1024**3:.2f} GiB')

## Run A2

This is evaluation-only. Candidate construction and strict-prior seen-history scans are Drive-bound; exact matrix retrieval then uses the GPU. Progress is printed throughout.

In [ ]:
command = [
    sys.executable, str(evaluation_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--v4-dir', str(V4_DIR),
    '--protocol-dir', str(PROTOCOL_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--split', 'validation',
    '--max-queries', '0',
    '--top-ks', '10,50,100,500,1000',
    '--popularity-blend-weight', '0.02',
    '--bootstrap-samples', '1000',
    '--item-batch-size', '8192',
    '--query-batch-size', '16',
    '--device', 'cuda',
]
log_path = Path('/content/adaptive_protocol_v2_a2.log')
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with log_path.open('w') as log_file:
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-160:])
    raise RuntimeError(f'A2 failed with exit code {return_code}. Last log lines:\n{tail}')

## Inspect the controlled comparison

In [ ]:
report = json.loads((OUTPUT_DIR / 'protocol_v2_evaluation.json').read_text())
summary = pd.read_csv(OUTPUT_DIR / 'summary.csv')
segments = pd.read_csv(OUTPUT_DIR / 'segments.csv')
display(summary[['variant', 'queries', 'users', 'hitrate@10', 'ndcg@10', 'hitrate@100', 'hitrate@1000']])
display(segments[segments['variant'].eq('v4_adaptive')][['segment_type', 'segment', 'queries', 'users', 'hitrate@10', 'ndcg@10', 'hitrate@1000']])
print('diagnostics:', report['diagnostics'])
for variant in summary['variant']:
    print(variant, report['clustered_user_bootstrap_ci95'][variant]['ndcg@10'])

## Save only lightweight reports

No item-embedding cache or checkpoint is copied to Drive.

In [ ]:
archive_path = Path('/content/adaptive_protocol_v2_a2_report.zip')
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, Path('protocol_v2_a2') / path.relative_to(OUTPUT_DIR))
    if log_path.exists():
        archive.write(log_path, 'logs/adaptive_protocol_v2_a2.log')
persistent_dir = DRIVE_ROOT / 'artifacts/adaptive/protocol_v2_a2'
persistent_dir.mkdir(parents=True, exist_ok=True)
persistent_path = persistent_dir / archive_path.name
shutil.copy2(archive_path, persistent_path)
print('saved:', persistent_path, archive_path.stat().st_size, 'bytes')
files.download(str(archive_path))